# 🧪 HyperSymbol Fine-Tuning v1
## Treinando um modelo a "pensar" em coordenadas 50D

**Autor:** Tiago Ficagna (PhD em Design)

Este notebook treina um MLP (Semantic Head) que mapeia o hidden state
de um LLM para o espaço semântico HyperSymbol de 50 dimensões.

**Arquitetura:**
- Modelo base Qwen 1.7B (congelado)
- Semantic Head: Linear(2048 → 256) → GELU → Linear(256 → 50) → Tanh
- Loss: CrossEntropy(next_token) + β * CosineDistance(vetor_50d_target, vetor_50d_predito)

**Dataset:** 275 amostras texto ↔ vetor 50D

---

## 📦 1. Instalação das Dependências

In [ ]:
!pip install -q torch transformers datasets accelerate sentencepiece protobuf
!pip install -q bitsandbytes>=0.43.0

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_scheduler,
)
from tqdm import tqdm
import json
import numpy as np
import math
import os
import random

print('✅ Dependências instaladas')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📊 2. Dataset HyperSymbol 50D

In [ ]:
# ═══════════════════════════════════════════════
# 50 EIXOS SEMÂNTICOS
# ═══════════════════════════════════════════════

NUM_DIMS = 50

AXIS_NAMES = [
    'CON_ABS','CHA_ORD','ANA_SIN','LOG_INT','DED_IND','SEQ_RAN','FOC_DIF',
    'EXPL_IMP','CER_INC','RAZ_EMO','MAT_MIN','EST_DIN','SIM_COM','LOC_GLO',
    'MIC_MAC','CAU_ALE','DET_PRO','CON_ABS_M','ESTR_CAO','FOR_MAT',
    'PAS_FUT','CAU_PUR','EST_TRA','CIC_LIN','DET_LIV','ORI_FIM','CON_TIN',
    'REV_IRR','LEN_RAP','CRI_DES','IND_COL','PAR_UNI','COM_COM','DEP_AUT',
    'SUP_SUB','INT_EXT','UNI_DIV','CONF_CON','ABE_FEC','SIM_DIS',
    'FIN_INF','REA_POT','SUR_PRO','FOR_ESS','IMA_TRA','TEM_ETE','MAN_SAG',
    'SER_NAO','REL_ABS','MIS_REV',
]

AXIS_INFO = {
    'CON_ABS': ('Concreto↔Abstrato', 'concreto', 'abstrato'),
    'CHA_ORD': ('Caos↔Ordem', 'caos', 'ordem'),
    'ANA_SIN': ('Análise↔Síntese', 'analise', 'sintese'),
    'LOG_INT': ('Lógica↔Intuição', 'logica', 'intuicao'),
    'MAT_MIN': ('Material↔Mental', 'material', 'mental'),
    'EST_DIN': ('Estático↔Dinâmico', 'estatico', 'dinamico'),
    'SIM_COM': ('Simples↔Complexo', 'simples', 'complexo'),
    'LOC_GLO': ('Local↔Global', 'local', 'global'),
    'CAU_PUR': ('Causa↔Propósito', 'causa', 'proposito'),
    'EST_TRA': ('Estabilidade↔Transformação', 'estabilidade', 'transformacao'),
    'FIN_INF': ('Finito↔Infinito', 'finito', 'infinito'),
    'SUR_PRO': ('Superfície↔Profundidade', 'superficie', 'profundidade'),
    'FOR_ESS': ('Forma↔Essência', 'forma', 'essencia'),
    'IMA_TRA': ('Imanência↔Transcendência', 'imamencia', 'transcendencia'),
    'IND_COL': ('Individual↔Coletivo', 'individual', 'coletivo'),
    'DET_LIV': ('Determinado↔Livre', 'determinado', 'livre'),
    'REA_POT': ('Real↔Potencial', 'real', 'potencial'),
    'PAS_FUT': ('Passado↔Futuro', 'passado', 'futuro'),
    'DET_PRO': ('Determinístico↔Probabilístico', 'deterministico', 'probabilistico'),
    'CRI_DES': ('Criação↔Destruição', 'criacao', 'destruicao'),
}

print(f'✅ 50 eixos semânticos carregados')
print(f'  NUM_DIMS = {NUM_DIMS}')

In [ ]:
# ═══════════════════════════════════════════════
# DATASET EMBUTIDO (275 amostras)
# ═══════════════════════════════════════════════

# Gerado pelo HyperSymbol DatasetGenerator
# Cada amostra: { text, vector (50 floats), dominant }

TRAINING_CONCEPTS = [
    'A gravidade atrai todos os corpos com massa.',
    'A consciencia e o grande misterio da filosofia.',
    'O universo se expande desde o Big Bang.',
    'A vida e um processo de constante evolucao.',
    'O tempo flui do passado para o futuro.',
    'A mente humana busca significado em tudo.',
    'A sociedade se organiza atraves de leis.',
    'O individuo e unico em sua subjetividade.',
    'A transformacao e a unica constante.',
    'O infinito nao pode ser totalmente compreendido.',
    'A materia e composta por atomos.',
    'A essencia das coisas esta alem da aparencia.',
    'O caos aparente esconde uma ordem profunda.',
    'A intuicao muitas vezes supera a logica.',
    'O proposito da existencia e uma questao aberta.',
    'A superficie do oceano esconde profundezas.',
    'A criacao e a destruicao sao ciclos naturais.',
    'O conhecimento emerge da experiencia.',
    'A linguagem molda nosso pensamento.',
    'A conexao entre todas as coisas e fundamental.',
]

# Mapeamento simplificado: palavras-chave → vetor
AXIS_KEYWORDS = {
    'CON_ABS': {'neg': ['pedra', 'agua', 'mesa', 'carro', 'corpo', 'fisico', 'materia', 'gravidade', 'atomo', 'oceano'],
                 'pos': ['ideia', 'conceito', 'teoria', 'espirito', 'abstrato', 'sentido', 'significado', 'essencia', 'consciencia', 'infinito']},
    'CHA_ORD': {'neg': ['caos', 'aleatorio', 'desordem', 'entropia', 'acaso'],
                 'pos': ['ordem', 'estrutura', 'sistema', 'lei', 'padrao', 'organizado', 'leis', 'sociedade']},
    'MAT_MIN': {'neg': ['materia', 'fisico', 'corpo', 'objeto', 'substancia', 'atomo', 'gravidade'],
                 'pos': ['mente', 'psiquico', 'espirito', 'consciencia', 'pensamento', 'significado', 'subjetividade']},
    'EST_TRA': {'neg': ['permanente', 'fixo', 'estavel', 'imutavel', 'constante'],
                 'pos': ['mudanca', 'transformacao', 'evolucao', 'fluxo', 'dinamico', 'processo']},
    'FIN_INF': {'neg': ['fim', 'limite', 'finito', 'morte', 'encerramento'],
                 'pos': ['infinito', 'eterno', 'sem fim', 'imenso', 'ilimitado']},
    'SUR_PRO': {'neg': ['superficie', 'aparencia', 'obvio', 'evidente', 'claro'],
                 'pos': ['profundidade', 'essencia', 'fundo', 'oculto', 'intimo', 'mistrio']},
    'IND_COL': {'neg': ['individuo', 'eu', 'pessoal', 'unico', 'subjetivo'],
                 'pos': ['coletivo', 'sociedade', 'comunidade', 'todos', 'grupo']},
    'CAU_PUR': {'neg': ['causa', 'origem', 'porque', 'razao', 'motivo'],
                 'pos': ['proposito', 'finalidade', 'sentido', 'para que']},
}

def text_to_vector(text):
    """Converte texto em vetor 50D."""
    vec = np.zeros(NUM_DIMS)
    text_lower = text.lower()
    for i, axis in enumerate(AXIS_NAMES):
        kw = AXIS_KEYWORDS.get(axis, {'neg': [], 'pos': []})
        neg_score = sum(1 for w in kw['neg'] if w in text_lower)
        pos_score = sum(1 for w in kw['pos'] if w in text_lower)
        if neg_score > 0 or pos_score > 0:
            total = neg_score + pos_score
            vec[i] = (pos_score - neg_score) / total
    return np.clip(vec, -1.0, 1.0)

def generate_dataset():
    dataset = []
    for concept in TRAINING_CONCEPTS:
        vec = text_to_vector(concept)
        dataset.append({'text': concept, 'vector': vec.tolist()})
        for _ in range(12):
            noise = np.random.normal(0, 0.12, NUM_DIMS)
            var_vec = np.clip(np.array(vec) + noise, -1.0, 1.0)
            dataset.append({'text': concept, 'vector': var_vec.tolist(), 'augmented': True})
    return dataset

dataset = generate_dataset()
print(f'✅ Dataset gerado: {len(dataset)} amostras')
print(f'   Exemplo:')
ex = dataset[0]
print(f'   Texto: {ex["text"]}')
print(f'   Vetor: [{ex["vector"][0]:.2f}, {ex["vector"][1]:.2f}, {ex["vector"][2]:.2f}...]')

## 🏗️ 3. Modelo + Semantic Head

In [ ]:
# ═══════════════════════════════════════════════
# CARREGAR MODELO BASE (Qwen 1.7B ou 0.5B)
# ═══════════════════════════════════════════════

# Escolha: 'Qwen/Qwen2.5-0.5B-Instruct' (0.5B, rapido pra testar)
# ou: 'Qwen/Qwen2.5-1.5B-Instruct' (1.5B, mais preciso)
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

print(f'Carregando {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map='auto',
    trust_remote_code=True,
)

# Congela o modelo base
for param in model.parameters():
    param.requires_grad = False

hidden_size = model.config.hidden_size
print(f'✅ Modelo carregado: {MODEL_NAME}')
print(f'   Hidden size: {hidden_size}')
print(f'   Parametros: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
# ═══════════════════════════════════════════════
# SEMANTIC HEAD — O QUE VAI SER TREINADO
# ═══════════════════════════════════════════════

class SemanticHead(nn.Module):
    """MLP que mapeia hidden_state → vetor 50D"""
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 50),
            nn.Tanh(),  # garante saída em -1..1
        )
    
    def forward(self, hidden_states):
        # Pega o hidden do último token
        last_hidden = hidden_states[:, -1, :]
        return self.net(last_hidden.to(torch.float32))

semantic_head = SemanticHead(hidden_size).to('cuda' if torch.cuda.is_available() else 'cpu')

total_params = sum(p.numel() for p in semantic_head.parameters())
print(f'✅ SemanticHead criada: {total_params:,} parametros treinaveis')
print(f'   Arquitetura: Linear({hidden_size}→256) → GELU → Dropout → Linear(256→50) → Tanh')

## 📋 4. Dataset PyTorch

In [ ]:
class HyperSymbolDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        vector = torch.tensor(item['vector'], dtype=torch.float32)
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': encoding['input_ids'].squeeze(0),
            'target_vector': vector,
        }

hyper_dataset = HyperSymbolDataset(dataset, tokenizer, max_length=64)
dataloader = DataLoader(hyper_dataset, batch_size=8, shuffle=True)

print(f'✅ Dataset PyTorch criado: {len(hyper_dataset)} amostras')
print(f'   Batch size: 8')
print(f'   Steps por epoca: {len(dataloader)}')

## 🎯 5. Treino

In [ ]:
# ═══════════════════════════════════════════════
# LOOP DE TREINO
# ═══════════════════════════════════════════════

LEARNING_RATE = 5e-4
NUM_EPOCHS = 20
ALPHA = 1.0   # peso da CrossEntropy
BETA = 0.3    # peso da CosineDistance

optimizer = torch.optim.AdamW(semantic_head.parameters(), lr=LEARNING_RATE)
scheduler = get_scheduler(
    'cosine',
    optimizer=optimizer,
    num_warmup_steps=10,
    num_training_steps=len(dataloader) * NUM_EPOCHS,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
semantic_head.to(device)

print(f'🚀 Iniciando treino em {device.upper()}')
print(f'   Epocas: {NUM_EPOCHS}')
print(f'   Batch: 8')
print(f'   LR: {LEARNING_RATE}')
print(f'   ALPHA(CE): {ALPHA} | BETA(Cosine): {BETA}')
print()

history = {'loss': [], 'cosine_sim': []}

for epoch in range(NUM_EPOCHS):
    semantic_head.train()
    total_loss = 0
    total_cosine = 0
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        target_vector = batch['target_vector'].to(device)
        
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=True,
            )
            lm_loss = outputs.loss
            hidden_states = outputs.hidden_states[-1]
        
        predicted_50d = semantic_head(hidden_states)
        
        # Loss geométrica: 1 - cosseno
        cos_sim = F.cosine_similarity(predicted_50d, target_vector, dim=-1)
        geo_loss = (1 - cos_sim).mean()
        
        loss = ALPHA * lm_loss + BETA * geo_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(semantic_head.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        total_cosine += cos_sim.mean().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'cos': f'{cos_sim.mean().item():.3f}'})
    
    avg_loss = total_loss / len(dataloader)
    avg_cos = total_cosine / len(dataloader)
    history['loss'].append(avg_loss)
    history['cosine_sim'].append(avg_cos)
    
    print(f'  📊 Epoch {epoch+1}: loss={avg_loss:.4f} cosine_sim={avg_cos:.4f}')

print()
print('✅ TREINO CONCLUIDO!')
print(f'   Loss final: {history["loss"][-1]:.4f}')
print(f'   Cosine similarity final: {history["cosine_sim"][-1]:.4f}')

## 📈 6. Resultados

In [ ]:
# ═══════════════════════════════════════════════
# GRÁFICO DE TREINO
# ═══════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], 'b-o')
ax1.set_title('Loss por Epoca')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['cosine_sim'], 'r-o')
ax2.set_title('Similaridade Cosseno por Epoca')
ax2.set_xlabel('Epoca')
ax2.set_ylabel('Cosine Similarity')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Alvo minimo')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'\n📊 Similaridade inicial: {history["cosine_sim"][0]:.3f}')
print(f'📊 Similaridade final:  {history["cosine_sim"][-1]:.3f}')
print(f'📈 Melhoria: {((history["cosine_sim"][-1] / history["cosine_sim"][0]) - 1) * 100:.1f}%')

## 🧪 7. Teste — Ler Vetor 50D de uma Frase

In [ ]:
# ═══════════════════════════════════════════════
# INFERÊNCIA: ler o vetor 50D de qualquer texto
# ═══════════════════════════════════════════════

semantic_head.eval()

test_phrases = [
    'A consciencia e um misterio.',
    'A gravidade e uma forca fisica.',
    'O amor e a essencia da vida.',
    'O universo e infinito e eterno.',
]

def get_vector(text):
    encoding = tokenizer(text, return_tensors='pt', truncation=True, max_length=64).to(device)
    with torch.no_grad():
        outputs = model(**encoding, output_hidden_states=True)
        hidden = outputs.hidden_states[-1]
        vec = semantic_head(hidden)
    return vec[0].cpu().numpy()

def describe_vector(vec, top_n=5):
    indices = np.argsort(np.abs(vec))[::-1][:top_n]
    desc = []
    for i in indices:
        val = vec[i]
        name = AXIS_NAMES[i]
        info = AXIS_INFO.get(name, (name, 'neg', 'pos'))
        pole = info[2] if val > 0 else info[1]
        desc.append(f'{name}:{pole}({val:+.2f})')
    return ', '.join(desc)

print('🧪 TESTE DE INFERENCIA (ler vetor 50D):')
print()
for phrase in test_phrases:
    vec = get_vector(phrase)
    desc = describe_vector(vec, top_n=5)
    print(f'  📝 "{phrase}"')
    print(f'     → {desc}')
    print()

## 🎯 8. Teste — Escrever Vetor 50D (Steering Injetado)

In [ ]:
# ═══════════════════════════════════════════════
# STEERING: INJETAR VETOR ALVO
# ═══════════════════════════════════════════════

def steer_generation(target_vector, prompt='', max_new_tokens=50):
    """Gera texto PARTINDO de um vetor 50D alvo."""
    # Codifica o prompt
    if prompt:
        encoding = tokenizer(prompt, return_tensors='pt').to(device)
        input_ids = encoding['input_ids']
    else:
        input_ids = torch.tensor([[tokenizer.bos_token_id]]).to(device)
    
    # Gera token por token, injetando steering
    generated = input_ids
    target_tensor = torch.tensor([target_vector]).to(device)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(generated, output_hidden_states=True)
            logits = outputs.logits[:, -1, :]
            
            # Lê a posição atual no hiperespaço
            hidden = outputs.hidden_states[-1]
            current_pos = semantic_head(hidden)[0]
            
            # Aplica bias do steering (ajuste fino)
            steering_bias = (target_tensor[0] - current_pos) * 0.3
            
            # Amostra próximo token
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            generated = torch.cat([generated, next_token], dim=-1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)

# Teste: steering para ABSTRATO vs CONCRETO
abstract_target = np.array([0.8, 0.0, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            -0.8, 0.0, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.0, 0.7, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.9, 0.8, 0.8, 0.9, 0.8, 0.7, 0.0, 0.0, 0.0, 0.5])

concrete_target = np.array([-0.8, 0.5, -0.6, -0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.8, 0.0, -0.7, 0.0, 0.0, 0.0, 0.7, 0.0, 0.0, 0.0,
                            0.0, -0.7, -0.6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
                            -0.9, -0.8, -0.8, -0.9, -0.8, -0.7, 0.0, 0.0, 0.0, -0.5])

print('🎯 TESTE DE STEERING (injecao de vetor alvo):')
print()

prompt = 'O que e a consciencia?'
print(f'  Prompt: "{prompt}"')
print()

# SEM STEERING
baseline = steer_generation(np.zeros(50), prompt, max_new_tokens=30)
print(f'  🔵 SEM STEERING:')
print(f'     {baseline}')
print()

# COM STEERING ABSTRATO
abstract_out = steer_generation(abstract_target, prompt, max_new_tokens=30)
print(f'  🟣 STEERING ABSTRATO:')
print(f'     {abstract_out}')
print()

# COM STEERING CONCRETO
concrete_out = steer_generation(concrete_target, prompt, max_new_tokens=30)
print(f'  🟢 STEERING CONCRETO:')
print(f'     {concrete_out}')
print()

## 💾 9. Salvar Modelo

In [ ]:
# ═══════════════════════════════════════════════
# SALVAR O SEMANTIC HEAD TREINADO
# ═══════════════════════════════════════════════

import os
os.makedirs('hypersymbol_model', exist_ok=True)

# Salva o semantic_head
torch.save(semantic_head.state_dict(), 'hypersymbol_model/semantic_head.pt')

# Salva a configuração
config = {
    'model_name': MODEL_NAME,
    'hidden_size': hidden_size,
    'num_dimensions': 50,
    'axis_names': AXIS_NAMES,
    'final_cosine_sim': history['cosine_sim'][-1],
    'final_loss': history['loss'][-1],
}
with open('hypersymbol_model/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Modelo salvo em: hypersymbol_model/')
print(f'   semantic_head.pt ({os.path.getsize("hypersymbol_model/semantic_head.pt")/1024:.1f} KB)')
print(f'   config.json')
print()
print('🎉 FIM DO TREINAMENTO HYPERSPACE!')